# 🔐 Fraud Detection — End-to-End Deep Learning Pipeline

**UAS Machine Learning — Individual Task**  
**Dataset:** IEEE-CIS Fraud Detection (`train_transaction.csv`)  

**Pipeline Overview:**
1. Data Loading & EDA
2. Data Preprocessing & Feature Engineering
3. Handling Class Imbalance (SMOTE)
4. Deep Learning Model (MLP with PyTorch)
5. Hyperparameter Tuning (Optuna)
6. Evaluation (ROC-AUC, F1, Confusion Matrix)
7. MLflow Tracking

---
**Name:** Muhammad Aqsandy  
**Class:** TK-46-02  
**NIM:** 1103220214

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install optuna mlflow imbalanced-learn torch torchvision --quiet

In [ ]:
# ============================================================
# CELL 2: Import Libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score)
# Imbalanced
from imblearn.over_sampling import SMOTE

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# MLflow
import mlflow
import mlflow.pytorch

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded. Device: {device}')
print(f'   PyTorch version: {torch.__version__}')

In [ ]:
# ============================================================
# CELL 3: Load Dataset
# ============================================================
# Kaggle IEEE-CIS Fraud Detection dataset
# Download from: https://www.kaggle.com/competitions/ieee-fraud-detection/data
# Or from course link (Transacation Dataset)
# Place train_transaction.csv in the same folder or Google Drive

# Option A — Colab from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/dataset/train_transaction.csv'

# Option B — Direct path (adjust as needed)
DATA_PATH = 'train_transaction.csv'

print('Loading dataset...')
df = pd.read_csv(DATA_PATH)
print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Memory usage: {df.memory_usage().sum() / 1e6:.1f} MB')
df.head()

In [ ]:
# ============================================================
# CELL 4: Exploratory Data Analysis (EDA)
# ============================================================
print('=== Basic Info ===')
print(f'Shape: {df.shape}')
print(f'\nTarget Distribution:')
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': fraud_counts, 'Percentage': fraud_pct.round(2)}))

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Not Fraud (0)', 'Fraud (1)'], fraud_counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution (Raw Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(fraud_counts.values, labels=['Not Fraud', 'Fraud'],
            colors=['steelblue', 'tomato'], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: isFraud', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n⚠️  Severe class imbalance detected → will apply SMOTE')

In [ ]:
# ============================================================
# CELL 5: EDA — Missing Values & Numeric Features
# ============================================================
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct%': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Pct%', ascending=False)
print(f'Columns with missing values: {len(missing_df)}')
print(missing_df.head(20))

# Numeric feature distribution (sample)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'isFraud'][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    df[col].dropna().hist(ax=axes[i], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frequency')
plt.suptitle('Numeric Feature Distributions (Sample)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 6: Preprocessing — Feature Engineering
# ============================================================
print('=== Preprocessing Pipeline ===')

# Step 1: Drop columns with >70% missing values
threshold = 0.70
high_missing = missing_pct[missing_pct > threshold * 100].index.tolist()
print(f'Step 1: Dropping {len(high_missing)} columns with >{threshold*100}% missing')
df_clean = df.drop(columns=high_missing)

# Step 2: Separate features and target
TARGET = 'isFraud'
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# Step 3: Encode categorical columns
print('Step 2: Encoding categorical columns...')
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f'   Categorical columns: {len(cat_cols)} → {cat_cols[:5]}...')
le = LabelEncoder()
for col in cat_cols:
    X[col] = X[col].astype(str)
    X[col] = le.fit_transform(X[col])

# Step 4: Fill remaining NaN with median
print('Step 3: Filling NaN with median...')
X = X.fillna(X.median(numeric_only=True))

print(f'\n✅ Clean feature matrix: {X.shape}')
print(f'   Features: {X.shape[1]}, Samples: {X.shape[0]:,}')

In [ ]:
# ============================================================
# CELL 7: Train/Val/Test Split + Scaling
# ============================================================
# Split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train size : {X_train.shape[0]:,} ({y_train.mean()*100:.2f}% fraud)')
print(f'Val size   : {X_val.shape[0]:,} ({y_val.mean()*100:.2f}% fraud)')
print(f'Test size  : {X_test.shape[0]:,} ({y_test.mean()*100:.2f}% fraud)')

# StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
print('\n✅ Scaling applied (StandardScaler)')

In [ ]:
# ============================================================
# CELL 8: Handle Class Imbalance with SMOTE
# ============================================================
print('Applying SMOTE on training set...')
print(f'Before SMOTE: {pd.Series(y_train).value_counts().to_dict()}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print(f'After  SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}')
print(f'New train size: {X_train_res.shape[0]:,}')

# Convert to PyTorch tensors
def to_tensors(X, y):
    return (torch.FloatTensor(X).to(device),
            torch.FloatTensor(y.values if hasattr(y,'values') else y).to(device))

X_tr_t, y_tr_t = to_tensors(X_train_res, pd.Series(y_train_res))
X_vl_t, y_vl_t = to_tensors(X_val_sc, y_val)
X_te_t, y_te_t = to_tensors(X_test_sc, y_test)

train_ds = TensorDataset(X_tr_t, y_tr_t)
val_ds   = TensorDataset(X_vl_t, y_vl_t)
print('\n✅ Tensors ready')

In [ ]:
# ============================================================
# CELL 9: Define MLP Model
# ============================================================
class FraudMLP(nn.Module):
    """Multi-Layer Perceptron for Binary Fraud Classification."""
    def __init__(self, input_dim, hidden_dims, dropout_rate):
        super(FraudMLP, self).__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ]
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    return total_loss / len(loader.dataset)


def evaluate(model, X_tensor, y_tensor, threshold=0.5):
    model.eval()
    with torch.no_grad():
        probs = model(X_tensor).cpu().numpy()
        y_true = y_tensor.cpu().numpy()
    preds = (probs >= threshold).astype(int)
    auc   = roc_auc_score(y_true, probs)
    f1    = f1_score(y_true, preds, zero_division=0)
    return auc, f1, probs, preds

print('✅ Model architecture defined')
# Quick test
INPUT_DIM = X_train_res.shape[1]
test_model = FraudMLP(INPUT_DIM, [256, 128, 64], 0.3).to(device)
print(test_model)
total_params = sum(p.numel() for p in test_model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# ============================================================
# CELL 10: Optuna Hyperparameter Tuning
# ============================================================
def objective(trial):
    # Hyperparameter search space
    n_layers   = trial.suggest_int('n_layers', 2, 4)
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
    dropout    = trial.suggest_float('dropout', 0.1, 0.5)
    lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024])

    hidden_dims = [hidden_dim // (2**i) for i in range(n_layers)]
    hidden_dims = [max(h, 32) for h in hidden_dims]

    model = FraudMLP(INPUT_DIM, hidden_dims, dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    for epoch in range(5):  # Quick 5-epoch search
        train_one_epoch(model, loader, optimizer, criterion)

    auc, _, _, _ = evaluate(model, X_vl_t, y_vl_t)
    return auc

print('Running Optuna hyperparameter search (20 trials)...')
study = optuna.create_study(direction='maximize',
                             study_name='fraud_mlp',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_params = study.best_params
print(f'\n✅ Best ROC-AUC (val): {study.best_value:.4f}')
print(f'Best params: {best_params}')

In [ ]:
# ============================================================
# CELL 11: Optuna Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trial_values = [t.value for t in study.trials]
best_so_far  = pd.Series(trial_values).cummax()
axes[0].plot(trial_values, 'o-', color='steelblue', alpha=0.6, label='Trial AUC')
axes[0].plot(best_so_far, 'r--', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial #')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Optuna Optimization History', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Param importance (simple)
params_df = pd.DataFrame([t.params for t in study.trials])
params_df['auc'] = trial_values
corr = params_df.corr()['auc'].drop('auc').abs().sort_values(ascending=True)
axes[1].barh(corr.index, corr.values, color='coral')
axes[1].set_xlabel('|Correlation with AUC|')
axes[1].set_title('Parameter Importance (Correlation)', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('optuna_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 12: Train Final Model with Best Hyperparameters
# ============================================================
n_layers   = best_params['n_layers']
hidden_dim = best_params['hidden_dim']
dropout    = best_params['dropout']
lr         = best_params['lr']
batch_size = best_params['batch_size']
EPOCHS     = 30

hidden_dims = [hidden_dim // (2**i) for i in range(n_layers)]
hidden_dims = [max(h, 32) for h in hidden_dims]

final_model = FraudMLP(INPUT_DIM, hidden_dims, dropout).to(device)
optimizer   = optim.Adam(final_model.parameters(), lr=lr, weight_decay=1e-5)
criterion   = nn.BCELoss()
scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
loader      = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

train_losses, val_aucs = [], []
best_val_auc = 0
best_state   = None

print(f'Training final model for {EPOCHS} epochs...')
print(f'Architecture: {hidden_dims}, Dropout: {dropout:.3f}, LR: {lr:.5f}')

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(final_model, loader, optimizer, criterion)
    val_auc, val_f1, _, _ = evaluate(final_model, X_vl_t, y_vl_t)
    scheduler.step(1 - val_auc)

    train_losses.append(train_loss)
    val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state   = {k: v.clone() for k, v in final_model.state_dict().items()}

    if epoch % 5 == 0:
        print(f'  Epoch {epoch:02d}/{EPOCHS} | Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}')

# Load best weights
final_model.load_state_dict(best_state)
print(f'\n✅ Training complete! Best Val AUC: {best_val_auc:.4f}')

In [ ]:
# ============================================================
# CELL 13: Training Curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(range(1, EPOCHS+1), train_losses, 'b-o', markersize=3, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training Loss Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS+1), val_aucs, 'g-o', markersize=3, label='Val AUC')
axes[1].axhline(y=best_val_auc, color='r', linestyle='--', label=f'Best: {best_val_auc:.4f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Validation AUC Curve', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training Progress — Fraud Detection MLP', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 14: Final Evaluation on Test Set
# ============================================================
test_auc, test_f1, test_probs, test_preds = evaluate(final_model, X_te_t, y_te_t)
y_true_np = y_test.values

precision = precision_score(y_true_np, test_preds, zero_division=0)
recall    = recall_score(y_true_np, test_preds, zero_division=0)

print('=' * 50)
print('       FINAL TEST SET RESULTS')
print('=' * 50)
print(f'  ROC-AUC   : {test_auc:.4f}')
print(f'  F1-Score  : {test_f1:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print('=' * 50)
print()
print(classification_report(y_true_np, test_preds, target_names=['Not Fraud', 'Fraud']))

In [ ]:
# ============================================================
# CELL 15: Evaluation Visualizations
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_true_np, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'])
axes[0].set_title(f'Confusion Matrix\n(F1={test_f1:.4f})', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_true_np, test_probs)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'MLP (AUC = {test_auc:.4f})')
axes[1].plot([0,1], [0,1], 'k--', linewidth=1, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Test Set', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.suptitle('Model Evaluation — Fraud Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 16: MLflow Experiment Tracking
# ============================================================
mlflow.set_experiment('fraud_detection_mlp')

with mlflow.start_run(run_name='best_mlp_model'):
    # Log hyperparameters
    mlflow.log_param('model_type', 'MLP')
    mlflow.log_param('n_layers', n_layers)
    mlflow.log_param('hidden_dim', hidden_dim)
    mlflow.log_param('hidden_dims', str(hidden_dims))
    mlflow.log_param('dropout', dropout)
    mlflow.log_param('learning_rate', lr)
    mlflow.log_param('batch_size', batch_size)
    mlflow.log_param('epochs', EPOCHS)
    mlflow.log_param('optimizer', 'Adam')
    mlflow.log_param('class_imbalance_method', 'SMOTE')
    mlflow.log_param('input_features', INPUT_DIM)

    # Log metrics
    mlflow.log_metric('test_roc_auc', test_auc)
    mlflow.log_metric('test_f1', test_f1)
    mlflow.log_metric('test_precision', precision)
    mlflow.log_metric('test_recall', recall)
    mlflow.log_metric('best_val_auc', best_val_auc)

    # Log per-epoch metrics
    for ep, (loss_v, auc_v) in enumerate(zip(train_losses, val_aucs)):
        mlflow.log_metric('train_loss', loss_v, step=ep+1)
        mlflow.log_metric('val_auc', auc_v, step=ep+1)

    # Log artifacts
    for img in ['eda_class_distribution.png', 'optuna_results.png',
                'training_curves.png', 'evaluation_results.png']:
        try:
            mlflow.log_artifact(img)
        except:
            pass

    # Log model
    mlflow.pytorch.log_model(final_model, artifact_path='model')

    run_id = mlflow.active_run().info.run_id
    print(f'✅ MLflow run logged! Run ID: {run_id}')
    print(f'   Experiment: fraud_detection_mlp')
    print(f'   Metrics logged: ROC-AUC={test_auc:.4f}, F1={test_f1:.4f}')

In [ ]:
# ============================================================
# CELL 17: Save Model
# ============================================================
torch.save({
    'model_state_dict': final_model.state_dict(),
    'hyperparams': best_params,
    'hidden_dims': hidden_dims,
    'input_dim': INPUT_DIM,
    'metrics': {
        'test_roc_auc': test_auc,
        'test_f1': test_f1,
        'test_precision': float(precision),
        'test_recall': float(recall)
    }
}, 'fraud_mlp_best.pth')
print('✅ Model saved as fraud_mlp_best.pth')

# Summary
print('\n' + '='*55)
print('         PIPELINE SUMMARY — FRAUD DETECTION')
print('='*55)
print(f'Dataset     : train_transaction.csv')
print(f'Features    : {INPUT_DIM}')
print(f'Model       : MLP (PyTorch) — {hidden_dims}')
print(f'Imbalance   : SMOTE oversampling')
print(f'Tuning      : Optuna TPE (20 trials)')
print(f'Tracking    : MLflow experiment logged')
print(f'Test AUC    : {test_auc:.4f}')
print(f'Test F1     : {test_f1:.4f}')
print(f'Test Prec.  : {precision:.4f}')
print(f'Test Recall : {recall:.4f}')
print('='*55)